#### What the code does
One-hot encoding: algorithms can’t read lists of strings, so we turn the baskets into a True/False table.
Apriori: finds all itemsets appearing in at least 30% of baskets. It relies on a neat trick: if {beer, diapers} is infrequent, any bigger set containing it must be too, so it skips them. This is the Apriori principle.
Rule generation: splits each frequent itemset into antecedent/consequent pairs and computes the metrics.
Filtering on lift: removes rules that look good only because the consequent is popular.

In [4]:
import pandas as pd
from itertools import permutations

# 1. Data: each list is one shopping basket
baskets = [
    ["bread", "milk"],
    ["bread", "diapers", "beer", "eggs"],
    ["milk", "diapers", "beer", "cola"],
    ["bread", "milk", "diapers", "beer"],
    ["bread", "milk", "diapers", "cola"],
    ["bread", "milk"],
    ["diapers", "beer"],
    ["bread", "milk", "eggs"],
    ["milk", "diapers", "beer"],
    ["bread", "diapers", "beer", "milk"],
]
n = len(baskets)
items = sorted({item for basket in baskets for item in basket})

# 2. Check every pair of items: A -> B
rows = []
for a, b in permutations(items, 2):
    count_a  = sum(a in basket for basket in baskets)
    count_b  = sum(b in basket for basket in baskets)
    count_ab = sum(a in basket and b in basket for basket in baskets)

    if count_ab == 0:
        continue

    support    = count_ab / n
    confidence = count_ab / count_a
    lift       = confidence / (count_b / n)

    rows.append({"rule": f"{a} -> {b}",
                 "support": support,
                 "confidence": confidence,
                 "lift": lift})

# 3. Put results in a table, best lift first
rules = pd.DataFrame(rows).sort_values("lift", ascending=False).round(3)
print(rules)

# 4. Save to Excel
rules.to_excel("t.xlsx", index=False)
print("Saved to rules.xlsx")

                rule  support  confidence   lift
2    beer -> diapers      0.6       1.000  1.429
20     eggs -> bread      0.2       1.000  1.429
14   diapers -> beer      0.6       0.857  1.429
12   cola -> diapers      0.2       1.000  1.429
16   diapers -> cola      0.2       0.286  1.429
8      bread -> eggs      0.2       0.286  1.429
13      cola -> milk      0.2       1.000  1.250
25      milk -> cola      0.2       0.250  1.250
9      bread -> milk      0.6       0.857  1.071
24     milk -> bread      0.6       0.750  1.071
26   milk -> diapers      0.5       0.625  0.893
18   diapers -> milk      0.5       0.714  0.893
10      cola -> beer      0.1       0.500  0.833
23      milk -> beer      0.4       0.500  0.833
19      eggs -> beer      0.1       0.500  0.833
1       beer -> cola      0.1       0.167  0.833
3       beer -> eggs      0.1       0.167  0.833
4       beer -> milk      0.4       0.667  0.833
7   bread -> diapers      0.4       0.571  0.816
15  diapers -> bread

C:\Users\Asus\AppData\Local\Temp\ipykernel_8252\3266669271.py:44: UserWarning: Pandas requires version '3.2.5' or newer of 'xlsxwriter' (version '3.2.0' currently installed).
  rules.to_excel("t.xlsx", index=False)


In [3]:
items

['beer', 'bread', 'cola', 'diapers', 'eggs', 'milk']

In [ ]:
%pip install mlxtend

In [11]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# 1. Data
baskets = [
    ["bread", "milk"],
    ["bread", "diapers", "beer", "eggs"],
    ["milk", "diapers", "beer", "cola"],
    ["bread", "milk", "diapers", "beer"],
    ["bread", "milk", "diapers", "cola"],
    ["bread", "milk"],
    ["diapers", "beer"],
    ["bread", "milk", "eggs"],
    ["milk", "diapers", "beer"],
    ["bread", "diapers", "beer", "milk"],
]

# 2. Convert to True/False table
te = TransactionEncoder()
df = pd.DataFrame(te.fit(baskets).transform(baskets), columns=te.columns_)

# 3. Find frequent itemsets and rules
frequent = apriori(df, min_support=0.2, use_colnames=True)
rules = association_rules(frequent, metric="confidence", min_threshold=0.6)

# 4. Keep the useful columns, make them readable, sort by lift
rules = rules[["antecedents", "consequents", "support", "confidence", "lift"]]
rules["antecedents"] = rules["antecedents"].apply(lambda s: ", ".join(s))
rules["consequents"] = rules["consequents"].apply(lambda s: ", ".join(s))
rules = rules.sort_values("lift", ascending=False).round(3)

print(rules)

# 5. Save to Excel
rules.to_excel("t.xlsx", index=False)

             antecedents    consequents  support  confidence   lift
21                  cola  diapers, milk      0.2       1.000  2.000
1                diapers           beer      0.6       0.857  1.429
3                   eggs          bread      0.2       1.000  1.429
6                   cola        diapers      0.2       1.000  1.429
10           beer, bread        diapers      0.3       1.000  1.429
23     beer, milk, bread        diapers      0.2       1.000  1.429
14            beer, milk        diapers      0.4       1.000  1.429
20            cola, milk        diapers      0.2       1.000  1.429
0                   beer        diapers      0.6       1.000  1.429
25           beer, bread  diapers, milk      0.2       0.667  1.333
15         diapers, milk           beer      0.4       0.800  1.333
16                  beer  diapers, milk      0.4       0.667  1.333
19         diapers, cola           milk      0.2       1.000  1.250
7                   cola           milk      0.2

C:\Users\Asus\AppData\Local\Temp\ipykernel_8252\1410329389.py:36: UserWarning: Pandas requires version '3.2.5' or newer of 'xlsxwriter' (version '3.2.0' currently installed).
  rules.to_excel("t.xlsx", index=False)


#### Stardarised Lift

Standardised lift rescales lift to a 0 to 1 range by comparing it with the smallest and largest lift that item pair could possibly have, given how often each item is bought (McNicholas et al., 2008).

The formulas

With sA = support of A, sB = support of B, and n = number of baskets:

Upper bound of lift: U = min(sA, sB) / (sA × sB)
Lower bound of lift: L = max(sA + sB − 1, 1/n) / (sA × sB)
Standardised lift = (lift − L) / (U − L)

In [15]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

baskets = [
    ["bread", "milk"],
    ["bread", "diapers", "beer", "eggs"],
    ["milk", "diapers", "beer", "cola"],
    ["bread", "milk", "diapers", "beer"],
    ["bread", "milk", "diapers", "cola"],
    ["bread", "milk"],
    ["diapers", "beer"],
    ["bread", "milk", "eggs"],
    ["milk", "diapers", "beer"],
    ["bread", "diapers", "beer", "milk"],
]
n = len(baskets)

te = TransactionEncoder()
df = pd.DataFrame(te.fit(baskets).transform(baskets), columns=te.columns_)

# Same thresholds as before: support 0.3, confidence 0.7
frequent = apriori(df, min_support=0.2, use_colnames=True)
rules = association_rules(frequent, metric="confidence", min_threshold=0.7)

# Standardised lift
sA, sB = rules["antecedent support"], rules["consequent support"]
rules["lift_lower"] = (sA + sB - 1).clip(lower=1/n) / (sA * sB)
rules["lift_upper"] = pd.concat([sA, sB], axis=1).min(axis=1) / (sA * sB)
rules["std_lift"] = (rules["lift"] - rules["lift_lower"]) / (rules["lift_upper"] - rules["lift_lower"])

# Filter on standardised lift instead of lift > 1
rules = rules[rules["std_lift"] >= 0.7]

# Clean up and save
cols = ["antecedents", "consequents", "support", "confidence",
        "lift", "lift_lower", "lift_upper", "std_lift"]
rules = rules[cols].copy()
rules["antecedents"] = rules["antecedents"].apply(lambda s: ", ".join(s))
rules["consequents"] = rules["consequents"].apply(lambda s: ", ".join(s))
rules = rules.sort_values("std_lift", ascending=False).round(3)

print(rules)
rules.to_excel("t.xlsx", index=False)

          antecedents    consequents  support  confidence   lift  lift_lower  \
13      diapers, cola           milk      0.2       1.000  1.250       0.625   
6                cola           milk      0.2       1.000  1.250       0.625   
1             diapers           beer      0.6       0.857  1.429       0.714   
0                beer        diapers      0.6       1.000  1.429       0.714   
15               cola  diapers, milk      0.2       1.000  2.000       1.000   
8         beer, bread        diapers      0.3       1.000  1.429       0.476   
10         beer, milk        diapers      0.4       1.000  1.429       0.357   
2                eggs          bread      0.2       1.000  1.429       0.714   
5                cola        diapers      0.2       1.000  1.429       0.714   
14         cola, milk        diapers      0.2       1.000  1.429       0.714   
16  beer, milk, bread        diapers      0.2       1.000  1.429       0.714   
11      diapers, milk           beer    

C:\Users\Asus\AppData\Local\Temp\ipykernel_8252\2172947521.py:44: UserWarning: Pandas requires version '3.2.5' or newer of 'xlsxwriter' (version '3.2.0' currently installed).
  rules.to_excel("t.xlsx", index=False)
